Some potential questions to ask:
- Do smaller videos (low views) have higher engagement rates than viral videos?
- Which categories generate the most engagement per view?
- Are comments more sensitive to virality than likes?
- Do engagement patterns differ across countries?

In [0]:
# Import dependencies
import plotly.express as px
import pandas as pd

In [0]:
# Load the "youtube_gold" table as a Spark DataFrame
df_gold = spark.table("youtube_gold")

In [0]:
# Count the number of rows and columns in the DataFrame
# Verify the size before converting to Pandas
row_count = df_gold.count()
col_count = len(df_gold.columns)

# Sample 1% of the data to estimate row size
sample_size_bytes = (df_gold
    .sample(fraction=0.01)
    .toPandas()
    .memory_usage(deep=True)
    .sum())

estimated_mb = (sample_size_bytes / 0.01) / 1_000_000

print(f"Rows:            {row_count:,}")
print(f"Columns:         {col_count}")
print(f"Estimated size:  {estimated_mb:.2f} MB")

In [0]:
# Convert the DataFrame to Pandas
df_gold = df_gold.toPandas()

In [0]:
# View df_gold info
df_gold.info()

In [0]:
# Create historgram for engagement rate
fig = px.histogram(
    df_gold,
    x="engagement_rate",
    title="Distribution of Engagement Rate"
)

fig.show()

In [0]:
# Engagement Rate v Log Views scatter plot
bins = [0, 100_000, 1_000_000, 10_000_000, float('inf')]
labels = ['<100K', '100K-1M', '1M-10M', '10M+']
df_gold['view_tier'] = pd.cut(df_gold['views'], bins=bins, labels=labels)

tier_medians = (
    df_gold.groupby('view_tier', observed=True)
    .agg({'engagement_rate': 'median', 'log_views': 'median'})
    .reset_index()
)

fig = px.scatter(
    df_gold,
    x='log_views',
    y='engagement_rate',
    color='view_tier',
    opacity=0.2,
    labels={'log_views': 'Log Views', 'engagement_rate': 'Engagement Rate'},
    title='Engagement Rate vs. Video Size'
)

fig.show()

In [0]:
# Engagement by category
# Exclude categories with insufficient sample size for meaningful analysis
min_rows = 50

category_counts = df_gold.groupby('category_name')['engagement_rate'].count()
valid_categories = category_counts[category_counts >= min_rows].index

category_stats = (
    df_gold[df_gold['category_name'].isin(valid_categories)]
    .groupby('category_name')['engagement_rate']
    .median()
    .reset_index()
    .sort_values('engagement_rate', ascending=True)
)

fig = px.bar(
    category_stats,
    x='engagement_rate',
    y='category_name',
    orientation='h',
    title='Median Engagement Rate by Category',
    labels={'engagement_rate': 'Median Engagement Rate', 'category_name': 'Category'}
)
fig.show()

In [0]:
# Comments v. Likes sensitivity to virality, dual-line
tier_stats = (
    df_gold.groupby('view_tier', observed=True)[['like_ratio', 'comment_rate']]
    .median()
    .reset_index()
    .melt(id_vars='view_tier', var_name='metric', value_name='rate')
)

fig = px.line(
    tier_stats,
    x='view_tier',
    y='rate',
    color='metric',
    markers=True,
    title='Like Rate vs. Comment Rate Across View Tiers',
    labels={'view_tier': 'View Tier', 'rate': 'Median Rate', 'metric': 'Metric'}
)
fig.show()

In [0]:
# Engagement by Region, heatmap
region_stats = (
    df_gold.groupby('region')[['engagement_rate', 'like_ratio', 'comment_rate']]
    .median()
    .reset_index()
)

fig = px.imshow(
    region_stats.set_index('region'),
    text_auto='.3f',
    aspect='auto',
    color_continuous_scale='Blues',
    title='Median Engagement Metrics by Region'
)
fig.show()

In [0]:
# Map to view engagement rate
region_stats = (
    df_gold.groupby('country')['engagement_rate']
    .median()
    .reset_index()
)

fig = px.choropleth(
    region_stats,
    locations='country',
    locationmode='country names',
    color='engagement_rate',
    title='Median Engagement Rate by Country',
    color_continuous_scale='Blues'
)

fig.update_layout(
    width=900,
    height=500,
    geo=dict(
        projection_type='natural earth'
    )
)

fig.show()